In [ ]:
!pip install jailbreakbench


In [ ]:
!pip install requests
import requests

In [ ]:
import requests
import pandas as pd
import jailbreakbench as jbb
import time
import random
import re

import os
GROQ_API_KEY = os.environ["GROQ_API_KEY"]  # export GROQ_API_KEY=... before running
API_URL = "https://api.groq.com/openai/v1/chat/completions"
HEADERS = {"Authorization": f"Bearer {GROQ_API_KEY}"}

SLEEP_TIME = 5  # rate limit safety

In [ ]:
import time
import requests

def call_llm(prompt: str, model_name: str, max_retries: int = 5) -> str:
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    wait_time = 5  # start with 5 seconds

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(API_URL, headers=HEADERS, json=payload).json()

            # ✅ Success
            if "choices" in r:
                return r["choices"][0]["message"]["content"]

            # 🔴 Rate limit or bad response
            print(
                f"[RATE LIMIT / ERROR] Attempt {attempt}/{max_retries} | "
                f"Waiting {wait_time}s"
            )
            print("Response:", r)

            time.sleep(wait_time)
            wait_time += 2  # 5 → 10 → 20 → 40 → 80

        except Exception as e:
            print(
                f"[EXCEPTION] Attempt {attempt}/{max_retries} | "
                f"Waiting {wait_time}s | Error: {e}"
            )
            time.sleep(wait_time)

In [ ]:
def call_llm(prompt: str, model_name: str) -> str:
  try:
    payload = {
      "model": model_name,
      "messages": [{"role": "user", "content": prompt}],
      "temperature": 0.7
    }

    r = requests.post(API_URL, headers=HEADERS, json=payload).json()

    if "choices" not in r:
      print("[ERROR] Bad LLM response:", r)
      time.sleep(5)
      r = requests.post(API_URL, headers=HEADERS, json=payload).json()
      if "choices" in r:
        return r["choices"][0]["message"]["content"]
      return ""

    return r["choices"][0]["message"]["content"]

  except Exception as e:
    print("[ERROR] LLM call failed:", e)
    return ""

In [ ]:
def run_attack(attack_name: str, model_name: str):
    print("\n" + "="*80)
    print(f"[START] Attack: {attack_name} | Model: {model_name}")
    print("="*80)

    # 1️⃣ Load JailbreakBench artifact
    print("[INFO] Loading JailbreakBench dataset...")

    if "pair" in attack_name.lower():
      model = "vicuna-13b-v1.5"
    else:
      model = "llama-2-7b-chat-hf"
    artifact = jbb.read_artifact(
        method = attack_name,
        model_name = model
    )

    rows = []
    for jb in artifact.jailbreaks:
        if jb.prompt is not None:
            rows.append({
                "index": jb.index,
                "goal": jb.goal,
                "category": jb.category,
                "prompt": jb.prompt
            })

    df = pd.DataFrame(rows)
    print(f"[INFO] Loaded {len(df)} jailbreak prompts")

    # 2️⃣ Run attack prompts
    responses = []
    total = len(df)

    print("[INFO] Running prompts against LLM...\n")

    for i, row in df.iterrows():
        print(f"[{i+1}/{total}] Running prompt (goal={row['goal']})")

        response = call_llm(row["prompt"], model_name)
        responses.append(response)

        # print("   ↳ Response received = ", response)
        time.sleep(SLEEP_TIME)

    df["llm_response"] = responses
    # df["attack"] = attack_name
    # df["model"] = model_name

    # 3️⃣ Save CSV
    safe_model_name = model_name.replace("/", "_")
    output_file = f"{attack_name}_{safe_model_name}.csv"

    df.to_csv(output_file, index=False)
    print(f"[DONE] Results saved to: {output_file}")

In [ ]:
# ATTACKS = ["GCG", "PAIR", "prompt_with_random_search"]
ATTACKS = ["PAIR"]

MODELS = [
    #"llama-3.1-8b-instant",
    #"meta-llama/llama-4-maverick-17b-128e-instruct",
    "qwen/qwen3-32b"
]

for attack in ATTACKS:
  for model in MODELS:
    run_attack(attack, model)